In [3]:
import duckdb
from huggingface_hub import hf_hub_download

con = duckdb.connect()

performance_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset"
)

content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset"
)

print("performance_file:", performance_file)
print("content_file:", content_file)


performance_file: C:\Users\anasm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance_sample.parquet
content_file: C:\Users\anasm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\dim_content.parquet


In [4]:
con.execute("PRAGMA memory_limit='1GB';")
con.execute("PRAGMA temp_directory='C:/Users/anasm/AppData/Local/Temp';")


In [46]:
con.execute(f"""
    CREATE OR REPLACE TABLE content_selected AS
    SELECT
        client_hash_id,
        content_hash_id,
        content_type,
        search_volume,
        competition,
        competition_level,
        cpc,
        main_intent,
        backlinks,
        category_count,
        char_count,
        word_count,
        last_optimized_date,
        optimization_eligible_date,
        is_published,
        is_deleted
    FROM read_parquet('{content_file}')
""")


In [6]:
con.execute("""
    SELECT COUNT(*) AS row_count
    FROM content_selected
""").df()


,row_count
0,519606


In [ ]:
con.execute("""
    DROP TABLE IF EXISTS performance_aggregated;
""")


In [ ]:
con.execute(f"""
    CREATE TABLE performance_aggregated AS
    SELECT
        client_hash_id,
        content_hash_id,

        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date,
        COUNT(DISTINCT report_date) AS reporting_days,

        SUM(gsc_impressions) AS total_gsc_impressions,
        SUM(gsc_clicks) AS total_gsc_clicks,
        SUM(gsc_sum_position) AS total_gsc_sum_position,
        AVG(gsc_avg_position) AS mean_gsc_avg_position

    FROM (
        SELECT DISTINCT *
        FROM read_parquet('{performance_file}')
    )
    GROUP BY
        client_hash_id,
        content_hash_id
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
con.execute("""
    SELECT COUNT(*) AS row_count
    FROM performance_aggregated
""").df()


,row_count
0,409205


In [13]:
con.execute("""
    DROP TABLE IF EXISTS analytical_dataset;
""")


In [14]:
con.execute("""
    CREATE TABLE analytical_dataset AS
    SELECT
        p.*,

        c.content_type,
        c.search_volume,
        c.competition,
        c.competition_level,
        c.cpc,
        c.main_intent,
        c.backlinks,
        c.category_count,
        c.char_count,
        c.word_count,
        c.last_optimized_date,
        c.optimization_eligible_date,
        c.is_published,
        c.is_deleted

    FROM performance_aggregated p
    LEFT JOIN content_selected c
        ON p.client_hash_id = c.client_hash_id
        AND p.content_hash_id = c.content_hash_id
""")


In [ ]:
con.execute("""
    SELECT COUNT(*) AS row_count
    FROM analytical_dataset
""").df()


,row_count
0,409205


In [16]:
overview = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_content_items,
        MIN(first_report_date) AS earliest_report_date,
        MAX(last_report_date) AS latest_report_date,
        AVG(reporting_days) AS avg_reporting_days
    FROM analytical_dataset
""").df()

overview

,total_rows,unique_clients,unique_content_items,earliest_report_date,latest_report_date,avg_reporting_days
0,409205,65,409205,2026-06-01,2026-06-30,28.561924


In [17]:
performance_summary = con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        MIN(total_gsc_impressions) AS min_impressions,
        AVG(total_gsc_impressions) AS avg_impressions,
        MEDIAN(total_gsc_impressions) AS median_impressions,
        MAX(total_gsc_impressions) AS max_impressions,

        MIN(total_gsc_clicks) AS min_clicks,
        AVG(total_gsc_clicks) AS avg_clicks,
        MEDIAN(total_gsc_clicks) AS median_clicks,
        MAX(total_gsc_clicks) AS max_clicks,

        MIN(mean_gsc_avg_position) AS min_avg_position,
        AVG(mean_gsc_avg_position) AS avg_avg_position,
        MEDIAN(mean_gsc_avg_position) AS median_avg_position,
        MAX(mean_gsc_avg_position) AS max_avg_position,

        MIN(reporting_days) AS min_reporting_days,
        AVG(reporting_days) AS avg_reporting_days,
        MEDIAN(reporting_days) AS median_reporting_days,
        MAX(reporting_days) AS max_reporting_days

    FROM analytical_dataset
""").df()

performance_summary

,total_rows,min_impressions,avg_impressions,median_impressions,max_impressions,min_clicks,avg_clicks,median_clicks,max_clicks,min_avg_position,avg_avg_position,median_avg_position,max_avg_position,min_reporting_days,avg_reporting_days,median_reporting_days,max_reporting_days
0,409205,0.0,528.184592,1.0,615012.0,0.0,2.954087,0.0,152170.0,0.0,22.929469,12.75,579.0,3,28.561924,30.0,30


In [ ]:

impression_buckets = con.execute("""
    SELECT
        CASE
            WHEN total_gsc_impressions = 0 THEN '0'
            WHEN total_gsc_impressions BETWEEN 1 AND 10 THEN '1-10'
            WHEN total_gsc_impressions BETWEEN 11 AND 100 THEN '11-100'
            WHEN total_gsc_impressions BETWEEN 101 AND 1000 THEN '101-1K'
            WHEN total_gsc_impressions BETWEEN 1001 AND 10000 THEN '1K-10K'
            WHEN total_gsc_impressions BETWEEN 10001 AND 100000 THEN '10K-100K'
            ELSE '100K+'
        END AS impression_bucket,

        COUNT(*) AS content_count,

        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS percentage

    FROM analytical_dataset

    GROUP BY impression_bucket

    ORDER BY
        CASE impression_bucket
            WHEN '0' THEN 1
            WHEN '1-10' THEN 2
            WHEN '11-100' THEN 3
            WHEN '101-1K' THEN 4
            WHEN '1K-10K' THEN 5
            WHEN '10K-100K' THEN 6
            WHEN '100K+' THEN 7
        END
""").df()

impression_buckets

,impression_bucket,content_count,percentage
0,0,200569,49.01
1,1-10,48993,11.97
2,11-100,58040,14.18
3,101-1K,66731,16.31
4,1K-10K,30866,7.54
5,10K-100K,3941,0.96
6,100K+,65,0.02


In [19]:
click_buckets = con.execute("""
    SELECT
        CASE
            WHEN total_gsc_clicks = 0 THEN '0'
            WHEN total_gsc_clicks BETWEEN 1 AND 5 THEN '1-5'
            WHEN total_gsc_clicks BETWEEN 6 AND 20 THEN '6-20'
            WHEN total_gsc_clicks BETWEEN 21 AND 100 THEN '21-100'
            WHEN total_gsc_clicks BETWEEN 101 AND 1000 THEN '101-1K'
            WHEN total_gsc_clicks BETWEEN 1001 AND 10000 THEN '1K-10K'
            ELSE '10K+'
        END AS click_bucket,

        COUNT(*) AS content_count,

        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS percentage

    FROM analytical_dataset

    GROUP BY click_bucket

    ORDER BY
        CASE click_bucket
            WHEN '0' THEN 1
            WHEN '1-5' THEN 2
            WHEN '6-20' THEN 3
            WHEN '21-100' THEN 4
            WHEN '101-1K' THEN 5
            WHEN '1K-10K' THEN 6
            WHEN '10K+' THEN 7
        END
""").df()

click_buckets

,click_bucket,content_count,percentage
0,0,325380,79.52
1,1-5,56262,13.75
2,6-20,18984,4.64
3,21-100,7420,1.81
4,101-1K,1131,0.28
5,1K-10K,26,0.01
6,10K+,2,0.00


In [21]:
content_type_distribution = con.execute("""
    SELECT
        COALESCE(content_type, 'Missing') AS content_type,
        COUNT(*) AS content_count,

        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS percentage

    FROM analytical_dataset

    GROUP BY content_type

    ORDER BY content_count DESC
""").df()

content_type_distribution

,content_type,content_count,percentage
0,keyword article,349328,85.37
1,feedly article,56485,13.80
2,comparison article,3392,0.83


In [22]:
main_intent_distribution = con.execute("""
    SELECT
        COALESCE(main_intent, 'Missing') AS main_intent,
        COUNT(*) AS content_count,

        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS percentage

    FROM analytical_dataset

    GROUP BY main_intent

    ORDER BY content_count DESC
""").df()

main_intent_distribution

,main_intent,content_count,percentage
0,informational,236625,57.83
1,Missing,71511,17.48
2,transactional,51882,12.68
3,commercial,47416,11.59
4,navigational,1771,0.43


In [23]:
competition_distribution = con.execute("""
    SELECT
        COALESCE(competition_level, 'Missing') AS competition_level,
        COUNT(*) AS content_count,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS percentage
    FROM analytical_dataset
    GROUP BY competition_level
    ORDER BY content_count DESC
""").df()

competition_distribution

,competition_level,content_count,percentage
0,LOW,284587,69.55
1,Missing,67831,16.58
2,HIGH,33804,8.26
3,MEDIUM,22983,5.62


In [24]:
content_size_summary = con.execute("""
    SELECT
        MIN(word_count) AS min_word_count,
        AVG(word_count) AS avg_word_count,
        MEDIAN(word_count) AS median_word_count,
        MAX(word_count) AS max_word_count,

        MIN(char_count) AS min_char_count,
        AVG(char_count) AS avg_char_count,
        MEDIAN(char_count) AS median_char_count,
        MAX(char_count) AS max_char_count
    FROM analytical_dataset
    WHERE word_count IS NOT NULL
      AND char_count IS NOT NULL
""").df()

content_size_summary

,min_word_count,avg_word_count,median_word_count,max_word_count,min_char_count,avg_char_count,median_char_count,max_char_count
0,0,2508.573792,2627.0,29341,0,16680.885741,17045.0,357575


In [25]:
content_size_quality = con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE word_count IS NULL
        ) AS missing_word_count,

        COUNT(*) FILTER (
            WHERE char_count IS NULL
        ) AS missing_char_count,

        COUNT(*) FILTER (
            WHERE word_count = 0
        ) AS zero_word_count,

        COUNT(*) FILTER (
            WHERE char_count = 0
        ) AS zero_char_count,

        COUNT(*) FILTER (
            WHERE word_count = 0
              AND char_count = 0
        ) AS both_zero
    FROM analytical_dataset
""").df()

content_size_quality

,total_rows,missing_word_count,missing_char_count,zero_word_count,zero_char_count,both_zero
0,409205,107758,107758,2,2,2


In [26]:
backlinks_summary = con.execute("""
    SELECT
        MIN(backlinks) AS min_backlinks,
        AVG(backlinks) AS avg_backlinks,
        MEDIAN(backlinks) AS median_backlinks,
        MAX(backlinks) AS max_backlinks,

        COUNT(*) FILTER (
            WHERE backlinks IS NULL
        ) AS missing_backlinks,

        COUNT(*) FILTER (
            WHERE backlinks = 0
        ) AS zero_backlinks
    FROM analytical_dataset
""").df()

backlinks_summary

,min_backlinks,avg_backlinks,median_backlinks,max_backlinks,missing_backlinks,zero_backlinks
0,0,476.573355,0.0,4269360,175929,183632


In [28]:
category_count_summary = con.execute("""
    SELECT
        MIN(category_count) AS min_category_count,
        AVG(category_count) AS avg_category_count,
        MEDIAN(category_count) AS median_category_count,
        MAX(category_count) AS max_category_count,

        COUNT(*) FILTER (
            WHERE category_count IS NULL
        ) AS missing_category_count,

        COUNT(*) FILTER (
            WHERE category_count = 0
        ) AS zero_category_count
    FROM analytical_dataset
""").df()

category_count_summary

,min_category_count,avg_category_count,median_category_count,max_category_count,missing_category_count,zero_category_count
0,0,1.203443,0.0,18,0,297163


In [29]:
performance_by_impressions = con.execute("""
    SELECT
        CASE
            WHEN total_gsc_impressions = 0 THEN '0'
            WHEN total_gsc_impressions <= 10 THEN '1-10'
            WHEN total_gsc_impressions <= 100 THEN '11-100'
            WHEN total_gsc_impressions <= 1000 THEN '101-1K'
            WHEN total_gsc_impressions <= 10000 THEN '1K-10K'
            WHEN total_gsc_impressions <= 100000 THEN '10K-100K'
            ELSE '100K+'
        END AS impression_bucket,

        COUNT(*) AS content_count,
        ROUND(AVG(total_gsc_clicks), 2) AS avg_clicks,
        MEDIAN(total_gsc_clicks) AS median_clicks
    FROM analytical_dataset
    GROUP BY impression_bucket
    ORDER BY
        CASE impression_bucket
            WHEN '0' THEN 1
            WHEN '1-10' THEN 2
            WHEN '11-100' THEN 3
            WHEN '101-1K' THEN 4
            WHEN '1K-10K' THEN 5
            WHEN '10K-100K' THEN 6
            WHEN '100K+' THEN 7
        END
""").df()

performance_by_impressions

,impression_bucket,content_count,avg_clicks,median_clicks
0,0,200569,0.00,0.0
1,1-10,48993,0.02,0.0
2,11-100,58040,0.34,0.0
3,101-1K,66731,1.60,1.0
4,1K-10K,30866,13.50,8.0
5,10K-100K,3941,85.95,47.0
6,100K+,65,5014.09,252.0


In [30]:
ctr_summary = con.execute("""
    SELECT
        COUNT(*) AS content_count,
        AVG(
            CASE
                WHEN total_gsc_impressions > 0
                THEN total_gsc_clicks * 1.0 / total_gsc_impressions
            END
        ) AS avg_ctr,
        MEDIAN(
            CASE
                WHEN total_gsc_impressions > 0
                THEN total_gsc_clicks * 1.0 / total_gsc_impressions
            END
        ) AS median_ctr,
        MAX(
            CASE
                WHEN total_gsc_impressions > 0
                THEN total_gsc_clicks * 1.0 / total_gsc_impressions
            END
        ) AS max_ctr
    FROM analytical_dataset
""").df()

ctr_summary

,content_count,avg_ctr,median_ctr,max_ctr
0,409205,0.007041,0.0,1.0


In [31]:
position_ctr_summary = con.execute("""
    SELECT
        CASE
            WHEN mean_gsc_avg_position IS NULL THEN 'Missing'
            WHEN mean_gsc_avg_position <= 3 THEN '1-3'
            WHEN mean_gsc_avg_position <= 5 THEN '4-5'
            WHEN mean_gsc_avg_position <= 10 THEN '6-10'
            WHEN mean_gsc_avg_position <= 20 THEN '11-20'
            ELSE '21+'
        END AS position_bucket,

        COUNT(*) AS content_count,

        ROUND(
            AVG(
                CASE
                    WHEN total_gsc_impressions > 0
                    THEN total_gsc_clicks * 1.0 / total_gsc_impressions
                END
            ),
            4
        ) AS avg_ctr,

        MEDIAN(
            CASE
                WHEN total_gsc_impressions > 0
                THEN total_gsc_clicks * 1.0 / total_gsc_impressions
            END
        ) AS median_ctr

    FROM analytical_dataset
    GROUP BY position_bucket
    ORDER BY
        CASE position_bucket
            WHEN '1-3' THEN 1
            WHEN '4-5' THEN 2
            WHEN '6-10' THEN 3
            WHEN '11-20' THEN 4
            WHEN '21+' THEN 5
            WHEN 'Missing' THEN 6
        END
""").df()

position_ctr_summary

,position_bucket,content_count,avg_ctr,median_ctr
0,1-3,13259,0.0079,0.000000
1,4-5,15341,0.0081,0.001339
2,6-10,59790,0.0053,0.000776
3,11-20,41795,0.0046,0.000517
4,21+,78451,0.0093,0.000000
5,Missing,200569,NaN,NaN


In [47]:
content_type_performance = con.execute("""
    SELECT
        content_type,
        COUNT(*) AS content_count,

        ROUND(AVG(total_gsc_impressions), 2) AS avg_impressions,
        MEDIAN(total_gsc_impressions) AS median_impressions,

        ROUND(AVG(total_gsc_clicks), 2) AS avg_clicks,
        MEDIAN(total_gsc_clicks) AS median_clicks,

        ROUND(
            AVG(
                CASE
                    WHEN total_gsc_impressions > 0
                    THEN total_gsc_clicks * 1.0 / total_gsc_impressions
                END
            ),
            4
        ) AS avg_ctr

    FROM analytical_dataset
    GROUP BY content_type
    ORDER BY content_count DESC
""").df()

content_type_performance

,content_type,content_count,avg_impressions,median_impressions,avg_clicks,median_clicks,avg_ctr
0,keyword article,349328,616.56,2.0,3.43,0.0,0.0047
1,feedly article,56485,10.82,0.0,0.21,0.0,0.0362
2,comparison article,3392,42.17,15.0,0.11,0.0,0.0026


In [33]:
intent_performance = con.execute("""
    SELECT
        COALESCE(main_intent, 'Missing') AS main_intent,
        COUNT(*) AS content_count,

        ROUND(AVG(total_gsc_impressions), 2) AS avg_impressions,
        MEDIAN(total_gsc_impressions) AS median_impressions,

        ROUND(AVG(total_gsc_clicks), 2) AS avg_clicks,
        MEDIAN(total_gsc_clicks) AS median_clicks,

        ROUND(
            AVG(
                CASE
                    WHEN total_gsc_impressions > 0
                    THEN total_gsc_clicks * 1.0 / total_gsc_impressions
                END
            ),
            4
        ) AS avg_ctr

    FROM analytical_dataset
    GROUP BY main_intent
    ORDER BY content_count DESC
""").df()

intent_performance

,main_intent,content_count,avg_impressions,median_impressions,avg_clicks,median_clicks,avg_ctr
0,informational,236625,585.68,2.0,3.12,0.0,0.0046
1,Missing,71511,21.67,0.0,0.30,0.0,0.0317
2,transactional,51882,815.43,17.0,6.03,0.0,0.0039
3,commercial,47416,705.27,10.0,2.78,0.0,0.0050
4,navigational,1771,142.63,0.0,1.91,0.0,0.0064


In [34]:
competition_performance = con.execute("""
    SELECT
        COALESCE(competition_level, 'Missing') AS competition_level,
        COUNT(*) AS content_count,

        ROUND(AVG(total_gsc_impressions), 2) AS avg_impressions,
        MEDIAN(total_gsc_impressions) AS median_impressions,

        ROUND(AVG(total_gsc_clicks), 2) AS avg_clicks,
        MEDIAN(total_gsc_clicks) AS median_clicks,

        ROUND(
            AVG(
                CASE
                    WHEN total_gsc_impressions > 0
                    THEN total_gsc_clicks * 1.0 / total_gsc_impressions
                END
            ),
            4
        ) AS avg_ctr

    FROM analytical_dataset
    GROUP BY competition_level
    ORDER BY
        CASE
            WHEN competition_level = 'LOW' THEN 1
            WHEN competition_level = 'MEDIUM' THEN 2
            WHEN competition_level = 'HIGH' THEN 3
            ELSE 4
        END
""").df()

competition_performance

,competition_level,content_count,avg_impressions,median_impressions,avg_clicks,median_clicks,avg_ctr
0,LOW,284587,668.45,3.0,3.25,0.0,0.0044
1,MEDIUM,22983,461.66,3.0,2.36,0.0,0.0044
2,HIGH,33804,385.99,2.0,6.07,0.0,0.0057
3,Missing,67831,33.10,0.0,0.34,0.0,0.0311


In [35]:
content_size_performance = con.execute("""
    SELECT
        CASE
            WHEN word_count IS NULL THEN 'Missing'
            WHEN word_count = 0 THEN '0'
            WHEN word_count <= 500 THEN '1-500'
            WHEN word_count <= 1000 THEN '501-1000'
            WHEN word_count <= 2000 THEN '1001-2000'
            WHEN word_count <= 3000 THEN '2001-3000'
            WHEN word_count <= 5000 THEN '3001-5000'
            ELSE '5000+'
        END AS word_count_bucket,

        COUNT(*) AS content_count,

        ROUND(AVG(total_gsc_impressions), 2) AS avg_impressions,
        MEDIAN(total_gsc_impressions) AS median_impressions,

        ROUND(AVG(total_gsc_clicks), 2) AS avg_clicks,
        MEDIAN(total_gsc_clicks) AS median_clicks,

        ROUND(
            AVG(
                CASE
                    WHEN total_gsc_impressions > 0
                    THEN total_gsc_clicks * 1.0 / total_gsc_impressions
                END
            ),
            4
        ) AS avg_ctr

    FROM analytical_dataset
    GROUP BY word_count_bucket
    ORDER BY
        CASE word_count_bucket
            WHEN '0' THEN 1
            WHEN '1-500' THEN 2
            WHEN '501-1000' THEN 3
            WHEN '1001-2000' THEN 4
            WHEN '2001-3000' THEN 5
            WHEN '3001-5000' THEN 6
            WHEN '5000+' THEN 7
            WHEN 'Missing' THEN 8
        END
""").df()

content_size_performance

,word_count_bucket,content_count,avg_impressions,median_impressions,avg_clicks,median_clicks,avg_ctr
0,0,2,1325.00,1325.0,6.50,6.5,0.0066
1,1-500,69,306.06,0.0,3.61,0.0,0.0058
2,501-1000,30589,31.46,0.0,0.15,0.0,0.0215
3,1001-2000,65854,198.93,0.0,1.05,0.0,0.0297
4,2001-3000,123124,1041.01,39.0,6.33,0.0,0.0048
5,3001-5000,73904,643.19,3.0,2.90,0.0,0.0043
6,5000+,7905,386.36,48.0,1.66,0.0,0.0047
7,Missing,107758,216.11,0.0,1.19,0.0,0.0044


In [ ]:
backlinks_performance = con.execute("""
    SELECT
        CASE
            WHEN backlinks IS NULL THEN 'Missing'
            WHEN backlinks = 0 THEN '0'
            WHEN backlinks <= 10 THEN '1-10'
            WHEN backlinks <= 100 THEN '11-100'
            WHEN backlinks <= 1000 THEN '101-1000'
            WHEN backlinks <= 10000 THEN '1001-10000'
            ELSE '10000+'
        END AS backlinks_bucket,

        COUNT(*) AS content_count,

        ROUND(AVG(total_gsc_impressions), 2) AS avg_impressions,
        MEDIAN(total_gsc_impressions) AS median_impressions,

        ROUND(AVG(total_gsc_clicks), 2) AS avg_clicks,
        MEDIAN(total_gsc_clicks) AS median_clicks,

        ROUND(
            AVG(
                CASE
                    WHEN total_gsc_impressions > 0
                    THEN total_gsc_clicks * 1.0 / total_gsc_impressions
                END
            ),
            4
        ) AS avg_ctr

    FROM analytical_dataset
    GROUP BY backlinks_bucket
    ORDER BY
        CASE backlinks_bucket
            WHEN '0' THEN 1
            WHEN '1-10' THEN 2
            WHEN '11-100' THEN 3
            WHEN '101-1000' THEN 4
            WHEN '1001-10000' THEN 5
            WHEN '10000+' THEN 6
            WHEN 'Missing' THEN 7
        END
""").df()

backlinks_performance

,backlinks_bucket,content_count,avg_impressions,median_impressions,avg_clicks,median_clicks,avg_ctr
0,0,183632,637.23,5.0,2.69,0.0,0.0045
1,1-10,7982,757.72,10.0,22.51,0.0,0.0066
2,11-100,16544,790.19,13.0,11.67,0.0,0.0047
3,101-1000,20029,732.24,17.0,2.75,0.0,0.0034
4,1001-10000,4438,779.06,20.0,3.03,0.0,0.0033
5,10000+,651,599.90,30.0,1.29,0.0,0.0018
6,Missing,175929,349.48,0.0,1.55,0.0,0.0123


In [42]:
from huggingface_hub import hf_hub_download

performance_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset"
)

print(performance_file)

C:\Users\anasm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance_sample.parquet


In [43]:
con.execute(f"""
    CREATE OR REPLACE VIEW performance_deduplicated AS
    SELECT DISTINCT *
    FROM read_parquet('{performance_file}')
""")

In [ ]:
daily_performance = con.execute("""
    SELECT
        report_date,
        COUNT(*) AS performance_rows,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position
    FROM performance_deduplicated
    GROUP BY report_date
    ORDER BY report_date
""").df()

daily_performance

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,performance_rows,total_impressions,total_clicks,avg_position
0,2026-06-01,390720,8926939.0,46668.0,18.595492
1,2026-06-02,391784,8843550.0,52160.0,19.634670
2,2026-06-03,392381,8497881.0,48339.0,19.452596
3,2026-06-04,392993,8374318.0,45435.0,19.691026
4,2026-06-05,393612,7397007.0,40214.0,19.036291
5,2026-06-06,394400,7351130.0,37558.0,19.885779
6,2026-06-07,395197,7927389.0,44007.0,19.442034
7,2026-06-08,395814,7495281.0,45060.0,19.068148
8,2026-06-09,396407,7410511.0,43498.0,17.796139
9,2026-06-10,397004,7435021.0,37297.0,17.902968


In [45]:
reporting_days_distribution = con.execute("""
    SELECT
        reporting_days,
        COUNT(*) AS content_count,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS percentage
    FROM analytical_dataset
    GROUP BY reporting_days
    ORDER BY reporting_days
""").df()

reporting_days_distribution

,reporting_days,content_count,percentage
0,3,2594,0.63
1,4,56,0.01
2,5,316,0.08
3,6,191,0.05
4,7,15,0.00
5,8,2937,0.72
6,9,17,0.00
7,10,378,0.09
8,11,317,0.08
9,12,17,0.00


### Finding 1 — Search Visibility Is Highly Uneven

**Finding:**

Search visibility is highly uneven across content items. Approximately 49.01% of content items received zero impressions, while the median number of impressions was only 1. A small number of content items received very large numbers of impressions, with a maximum of 615,012.

**Evidence:**

- Zero impressions: 200,569 / 409,205 content items (49.01%)
- Median impressions: 1
- Mean impressions: 528.18
- Maximum impressions: 615,012

**Insight:**

Search exposure is concentrated in a relatively small portion of content items. This indicates that raw impression counts are highly right-skewed and may require transformation or normalization when used as inputs to an opportunity-scoring model.

**ML Relevance:**

Impression volume should not be treated as a normally distributed feature. The strong skew and large difference between the median and mean should be considered during feature engineering and model development.

### Finding 2 — Click Activity Is Highly Sparse

**Finding:**

Click activity is highly sparse across content items. Approximately 79.52% of content items received zero clicks during the reporting period, and 93.27% received five clicks or fewer.

**Evidence:**

- Zero clicks: 325,380 / 409,205 content items (79.52%)
- Median clicks: 0
- Content items with 5 clicks or fewer: 93.27%
- Maximum clicks: 152,170

**Insight:**

Most content items generate little or no observed click activity, while a small number account for substantially larger click volumes. This creates a strongly skewed outcome distribution.

**ML Relevance:**

Click-based features or outcomes should account for the large concentration of zero values and the presence of extreme observations.

### Finding 3 — CTR Alone Does Not Capture Performance

**Finding:**

CTR varies substantially across content groups, but higher CTR values can occur in groups with very low impression exposure.

**Evidence:**

- Overall average CTR: approximately 0.704%
- Median CTR: 0
- Missing main-intent group: 3.17% average CTR with 21.67 average impressions
- Feedly articles: 3.62% average CTR with 10.82 average impressions

**Insight:**

CTR can be highly sensitive to low impression volumes. Therefore, a high CTR does not necessarily indicate high search traffic or strong overall performance.

**ML Relevance:**

CTR should not be used as the sole indicator when defining content opportunity. Exposure-related metrics such as impressions should also be considered when constructing the scoring framework.

### Finding 4 — Content Characteristics Are Associated With Different Performance Patterns

**Finding:**

Content performance distributions vary across several content-level characteristics, including content type, search intent, competition level, content size, and backlink volume.

**Evidence:**

- Keyword articles have substantially higher average impressions than Feedly and comparison articles.
- Transactional content has the highest average impressions among the main intent categories.
- Low-competition content has the highest average impression volume among records with known competition levels.
- The 2,001–3,000 word group has the highest average impression volume among the main content-size groups.
- Content with some backlink coverage generally shows higher median impression volumes than content with zero backlinks.

**Insight:**

Different content characteristics are associated with different search-performance distributions. However, the relationships are not consistently monotonic, so no single characteristic explains content performance on its own.

**ML Relevance:**

These content characteristics are potential candidate features for the opportunity-scoring model. Their individual relationships should be evaluated together rather than using any single feature as a standalone decision rule.

### Finding 5 — Reporting Exposure Varies Across Content

**Finding:**

Reporting coverage is uneven across content items, although most content items have near-complete coverage.

**Evidence:**

- 77.97% of content items have 27 reporting days.
- 14.57% have 25 reporting days.
- 2.28% have 29 reporting days.
- A small proportion has substantially fewer reporting days.

**Insight:**

Content items do not all have the same amount of observed performance exposure. Therefore, differences in total impressions or clicks may partly reflect differences in reporting coverage.

**ML Relevance:**

`reporting_days` should be considered as an exposure-related feature when constructing and validating the opportunity-scoring model.

### Task 6.4 — First Findings Summary

The EDA identified several patterns that are relevant to the Content Opportunity Scoring problem.

Search impressions and clicks are highly skewed, with a large proportion of content items receiving little or no observed search activity. CTR also shows strong skew and can be misleading when impression exposure is very low.

Content-level characteristics such as content type, main intent, competition level, content size, and backlink volume are associated with different performance distributions, but none of these characteristics shows a simple relationship that can independently explain content performance.

The available reporting period also shows variation in daily performance and reporting coverage. Therefore, exposure and reporting coverage should be considered when comparing content items.

These findings support the next phase of the project: defining the ML problem and designing features that can represent content performance, exposure, and content-level characteristics without making causal claims.